# 01 · ETL — Seca e Cobertura Hídrica (CE / PB · EPSG:31984)

**Fontes de dados**
- Reservatórios — ANA: SAR (série de volumes) + camadas do SNIRH (coordenadas). Substitui o Hidroweb, que não traz volume de açudes.
- Municípios CE e PB — GeoJSON do IBGE (API de malhas).
- População rural por município — Censo IBGE 2022 (SIDRA, tabela 9923).

**Etapas**
1. Download dos dados de volume de açudes da ANA
2. Reprojeção para EPSG:31984; join com municípios CE/PB
3. Cálculo de volume armazenado (%) por reservatório e mês
4. Exportar `reservatorios-ne.geojson` e `municipios-ne.geojson` (em `data/processed/`)

Os dados brutos baixados ficam em `data/raw/`. Requer acesso à internet; o servidor da ANA é instável (o notebook refaz as tentativas e leva alguns minutos).

In [1]:
import re, time, requests, pandas as pd, geopandas as gpd
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
from shapely.geometry import Point

CRS = "EPSG:31984"                       # SIRGAS 2000 / UTM 24S
UFS = {"CE": 23, "PB": 25}               # código IBGE da UF
SAR_UF = {"CE": 7, "PB": 16}             # código da UF no SAR
INI, FIM = "01/01/2020", "30/09/2026"
RAW, OUT = Path("../data/raw"), Path("../data/processed")
RAW.mkdir(parents=True, exist_ok=True); OUT.mkdir(parents=True, exist_ok=True)

S = requests.Session()
def get(url, **kw):
    """GET com novas tentativas (o SAR da ANA responde 5xx de vez em quando)."""
    for i in range(7):
        try:
            r = S.get(url, timeout=120, **kw); r.raise_for_status(); return r
        except Exception:
            if i == 6: raise
            time.sleep(4 * (i + 1))

## 1. Download — reservatórios (ANA)

**1a. Cadastro e coordenadas.** Camada *SAR Nordeste* do SNIRH (CE e PB).

In [2]:
r = get("https://www.snirh.gov.br/arcgis/rest/services/IG/SAR/FeatureServer/1/query",
        params={"where": "SAR_SG_UF IN ('CE','PB')", "outFields": "*", "outSR": 4326, "f": "geojson"})
res = gpd.GeoDataFrame.from_features(r.json()["features"], crs="EPSG:4326")
res["fonte_coord"] = "ANA SAR Nordeste"
print(len(res), "açudes na camada SAR Nordeste:", res.SAR_SG_UF.value_counts().to_dict())

260 açudes na camada SAR Nordeste: {'CE': 141, 'PB': 119}


**1b. Açudes monitorados no SAR que não estão nessa camada.** O SAR lista 21 açudes de CE/PB sem coordenadas na camada *SAR Nordeste*. Para 19 deles, a camada *Massas d'Água* (ligada pelo `cod_sar`) traz o polígono do açude, e usamos um ponto dentro dele. `CUREMA/MÃE D'ÁGUA` usa o ponto do açude Mãe d'Água (Coremas) da camada *Semiárido*. `BREJINHO` (SAR 12503) fica em Triunfo-PE e é descartado por estar fora de CE/PB.

In [3]:
EXTRA = {12063: ("BONITO", "CE"), 12073: ("BROCÓ", "CE"), 12151: ("DESTERRO", "CE"), 12153: ("DO CORONEL", "CE"),
         12527: ("ESCURIDÃO", "CE"), 12181: ("FAVELAS", "CE"), 12188: ("FORQUILHA II", "CE"), 12198: ("GERARDO ATIMBONE", "CE"),
         12225: ("JATOBÁ", "CE"), 12244: ("JUNCO", "CE"), 12260: ("MADEIRO", "CE"), 12536: ("MONTE BELO", "CE"),
         12398: ("SANTA MARIA DE ARACATIAÇU", "CE"), 12483: ("VÁRZEA DO BOI", "CE"), 12124: ("CHUPADOURO I", "PB"),
         12139: ("CRUZ DOS POCINHOS", "PB"), 12278: ("MILHÃ (E. GONÇALVES)", "PB"), 12294: ("NOVO II", "PB"),
         12481: ("VÁRZEA", "PB")}

r2 = get("https://www.snirh.gov.br/arcgis/rest/services/SPR/Massa_dagua/FeatureServer/0/query",
         params={"where": "cod_sar IN (%s)" % ",".join(map(str, EXTRA)), "outFields": "cod_sar,nmriocomp", "outSR": 4326, "f": "geojson"}).json()
ex = gpd.GeoDataFrame.from_features(r2["features"], crs="EPSG:4326").dissolve(by="cod_sar", as_index=False)
ex["geometry"] = ex.to_crs(CRS).representative_point().to_crs("EPSG:4326")
ex["SAR_ID"] = ex.cod_sar
ex["SAR_NM"] = ex.SAR_ID.map(lambda i: EXTRA[i][0]); ex["SAR_SG_UF"] = ex.SAR_ID.map(lambda i: EXTRA[i][1])
ex["SAR_NM_RIO"] = ex.nmriocomp; ex["SAR_NM_BACIA"] = None
ex["fonte_coord"] = "ANA Massas d'Água (ponto no polígono)"

r3 = get("https://www.snirh.gov.br/arcgis/rest/services/SPR/Semiarido_204/FeatureServer/0/query",
         params={"where": "NMRESERV LIKE '%gua' AND NMMUN='Coremas'", "outFields": "*", "outSR": 4326, "f": "geojson"}).json()
fx = r3["features"][0]
curema = gpd.GeoDataFrame([{"SAR_ID": 12144, "SAR_NM": "CUREMA/MÃE D'ÁGUA", "SAR_SG_UF": "PB", "SAR_NM_RIO": fx["properties"]["NMRIO"],
        "SAR_NM_BACIA": fx["properties"]["NMBACIA"], "fonte_coord": "ANA Semiárido 204 (açude Mãe d'Água, Coremas)",
        "geometry": Point(fx["geometry"]["coordinates"][0])}], crs="EPSG:4326")

cols = ["SAR_ID", "SAR_NM", "SAR_SG_UF", "SAR_NM_RIO", "SAR_NM_BACIA", "fonte_coord", "geometry"]
res = gpd.GeoDataFrame(pd.concat([res, ex[cols], curema[cols]], ignore_index=True), crs="EPSG:4326")
print(len(res), "açudes:", res.SAR_SG_UF.value_counts().to_dict())

280 açudes: {'CE': 155, 'PB': 125}


**1c. Série de volumes.** Consulta à página `/sar0/Medicao` do SAR, um açude por vez (a ANA não oferece API/CSV).

In [4]:
ROW = re.compile(r"<tr>(.*?)</tr>", re.S)
COL = re.compile(r'coluna_(\d)">\s*([^<]*?)\s*</td>')
num = lambda s: float(s.replace(".", "").replace(",", ".")) if s not in ("", "-") else None

def serie(item):
    sid, uf = item
    html = get("https://www.ana.gov.br/sar0/Medicao", params={"dropDownListEstados": SAR_UF[uf], "dropDownListReservatorios": sid,
               "dataInicial": INI, "dataFinal": FIM, "button": "Buscar"}).text
    rows = []
    for tr in ROW.findall(html):
        c = dict(COL.findall(tr))
        if "7" in c:    # 4 = capacidade (hm³), 3 = cota, 5 = volume (hm³), 6 = volume (%), 7 = data
            rows.append({"SAR_ID": sid, "capacidade_hm3": num(c["4"]), "volume_hm3": num(c["5"]),
                         "volume_pct_ana": num(c["6"]), "data": c["7"]})
    return rows

with ThreadPoolExecutor(4) as pool:
    med = pd.DataFrame([x for rows in pool.map(serie, zip(res.SAR_ID, res.SAR_SG_UF)) for x in rows])
med["data"] = pd.to_datetime(med.data, format="%d/%m/%Y")
med.to_csv(RAW / "ana_medicoes_brutas.csv", index=False)
res.to_file(RAW / "ana_reservatorios_ce_pb.geojson", driver="GeoJSON")
print(med.shape, med.SAR_ID.nunique(), "açudes com medições;", med.data.min().date(), "→", med.data.max().date())

(534308, 5) 279 açudes com medições; 2020-01-01 → 2026-09-30


## 2. Download — municípios CE/PB (IBGE) e população rural (Censo 2022)

In [5]:
feats = []
for uf, cod in UFS.items():
    g = get(f"https://servicodados.ibge.gov.br/api/v3/malhas/estados/{cod}",
            params={"formato": "application/vnd.geo+json", "intrarregiao": "municipio", "qualidade": "maxima"}).json()
    for f in g["features"]:
        f["properties"]["uf"] = uf
    feats += g["features"]
mun = gpd.GeoDataFrame.from_features(feats, crs="EPSG:4326").rename(columns={"codarea": "cod_mun"})
nomes = pd.DataFrame([{"cod_mun": str(m["id"]), "municipio": m["nome"]} for cod in UFS.values()
        for m in get(f"https://servicodados.ibge.gov.br/api/v1/localidades/estados/{cod}/municipios").json()])
mun = mun.merge(nomes, on="cod_mun")

# SIDRA 9923, variável 93 (população residente) por situação do domicílio (c1)
s = get("https://apisidra.ibge.gov.br/values/t/9923/n6/in%20n3%2023,25/v/93/p/2022/c1/all").json()[1:]
pop = pd.DataFrame(s)[["D1C", "D4N", "V"]].rename(columns={"D1C": "cod_mun", "D4N": "sit", "V": "v"})
pop["v"] = pd.to_numeric(pop.v, errors="coerce")
pop = pop.pivot(index="cod_mun", columns="sit", values="v").reset_index()
pop = pop.rename(columns={"Rural": "pop_rural", "Total": "pop_total", "Urbana": "pop_urbana"})
pop["pop_rural"] = pop.pop_rural.fillna(0)          # "-" no SIDRA = sem população rural

mun.to_file(RAW / "ibge_municipios_ce_pb.geojson", driver="GeoJSON")
pop.to_csv(RAW / "ibge_pop_rural_2022.csv", index=False)
print(len(mun), "municípios;", int(pop.pop_rural.sum()), "habitantes rurais")

407 municípios; 2845059 habitantes rurais


## 3. Reprojeção (EPSG:31984) e join com municípios CE/PB

Cada açude recebe o município do IBGE em que está localizado; a população rural é anexada ao polígono do município.

In [6]:
resp = res.to_crs(CRS)
munp = mun.merge(pop, on="cod_mun", how="left").to_crs(CRS)
resp = gpd.sjoin(resp, munp[["cod_mun", "municipio", "geometry"]], how="left", predicate="within").drop(columns="index_right")

fora = resp.cod_mun.isna()                       # ponto fora de polígono: usa o município mais próximo
if fora.any():
    j = gpd.sjoin_nearest(resp.loc[fora, ["geometry"]], munp[["cod_mun", "municipio", "geometry"]], how="left")
    j = j[~j.index.duplicated()]
    resp.loc[fora, ["cod_mun", "municipio"]] = j[["cod_mun", "municipio"]].values
print("açudes fora de polígono:", int(fora.sum()), "| CRS:", resp.crs.to_string())

açudes fora de polígono: 0 | CRS: EPSG:31984


## 4. Volume armazenado (%) por reservatório e mês

`volume_pct` = média mensal do volume ÷ capacidade × 100, limitado a 0–100 (a ANA tem registros com volume acima da capacidade).

In [7]:
med["mes"] = med.data.dt.to_period("M").dt.strftime("%Y-%m")
m = (med.groupby(["SAR_ID", "mes"])
        .agg(volume_hm3=("volume_hm3", "mean"), pct_ana=("volume_pct_ana", "mean"), n_medicoes=("data", "size")).reset_index())

capacidade_sar = med.groupby("SAR_ID").capacidade_hm3.agg(lambda x: x.mode().iloc[0])
capacidade_camada = resp.set_index("SAR_ID").SAR_QT_CAPACIDADE
m["capacidade_hm3"] = m.SAR_ID.map(capacidade_camada).fillna(m.SAR_ID.map(capacidade_sar))   # camada; se faltar, a do SAR
m["volume_pct"] = (m.volume_hm3 / m.capacidade_hm3 * 100).clip(0, 100).round(2)
m["volume_hm3"] = m.volume_hm3.round(3)

dif = (m.volume_pct - m.pct_ana).abs()
print("açudes com capacidade (camada × SAR) diferindo >5%:", int(((capacidade_camada.reindex(capacidade_sar.index) - capacidade_sar).abs() / capacidade_sar > .05).sum()))
print(f"diferença entre volume_pct e o % da ANA: média {dif.mean():.2f} pp; {(dif > 5).sum()} de {len(m)} linhas > 5 pp (registros da ANA com volume > capacidade)")
m = m.drop(columns="pct_ana")

base = resp[["SAR_ID", "SAR_NM", "SAR_SG_UF", "SAR_NM_BACIA", "SAR_NM_RIO", "fonte_coord", "cod_mun", "municipio", "geometry"]].rename(
    columns={"SAR_NM": "reservatorio", "SAR_SG_UF": "uf", "SAR_NM_BACIA": "bacia", "SAR_NM_RIO": "rio"})
vol = base.merge(m, on="SAR_ID")
uf_ibge = vol.drop_duplicates("SAR_ID").pipe(lambda d: d.cod_mun.str[:2].map({"23": "CE", "25": "PB"}) != d.uf)
print(vol.shape, "|", vol.SAR_ID.nunique(), "açudes com série | UF do SAR diferente da do IBGE:", int(uf_ibge.sum()))
vol.drop(columns="geometry").head()

açudes com capacidade (camada × SAR) diferindo >5%: 0
diferença entre volume_pct e o % da ANA: média 1.12 pp; 519 de 20703 linhas > 5 pp (registros da ANA com volume > capacidade)
(20703, 14) | 279 açudes com série | UF do SAR diferente da do IBGE: 0


,SAR_ID,reservatorio,uf,bacia,rio,fonte_coord,cod_mun,municipio,mes,volume_hm3,n_medicoes,capacidade_hm3,volume_pct
0,12005,ACAUÃ,PB,ALTO PARAÍBA,RIO PARAÍBA,ANA SAR Nordeste,2509909,Natuba,2020-01,22.895,31,253.0,9.05
1,12005,ACAUÃ,PB,ALTO PARAÍBA,RIO PARAÍBA,ANA SAR Nordeste,2509909,Natuba,2020-02,21.993,29,253.0,8.69
2,12005,ACAUÃ,PB,ALTO PARAÍBA,RIO PARAÍBA,ANA SAR Nordeste,2509909,Natuba,2020-03,23.491,31,253.0,9.28
3,12005,ACAUÃ,PB,ALTO PARAÍBA,RIO PARAÍBA,ANA SAR Nordeste,2509909,Natuba,2020-04,32.517,30,253.0,12.85
4,12005,ACAUÃ,PB,ALTO PARAÍBA,RIO PARAÍBA,ANA SAR Nordeste,2509909,Natuba,2020-05,33.571,31,253.0,13.27


## 5. Exportação

In [8]:
vol.to_file(OUT / "reservatorios-ne.geojson", driver="GeoJSON")
munp.to_file(OUT / "municipios-ne.geojson", driver="GeoJSON")
for f in ["reservatorios-ne.geojson", "municipios-ne.geojson"]:
    g = gpd.read_file(OUT / f)
    print(f, g.shape, g.crs.to_string())

reservatorios-ne.geojson (20703, 14) EPSG:31984
municipios-ne.geojson (407, 7) EPSG:31984
